# 04_1 — Generative Inversion with Vec2Text

Vec2Text reconstructs the original sentence directly from its embedding, iteratively refining a candidate at each step.

Unlike probing or classifier-based approaches, this produces **full text** — not just attributes.

**Prerequisite** : `00_1` and `00_2` must have been run. The GTR-encoded target index (`data/sentences_target_gtr_vector_db.index`) must also be available — it is provided as a data file.

**Setup.** Configure the inversion parameters. `NUM_STEPS` controls how many refinement iterations vec2text runs — more steps means higher fidelity but more compute time. Start with 20.

In [ ]:
from time import perf_counter

from gtr_runtime import load_gtr_encoder
from tqdm.notebook import tqdm
import faiss
import pandas as pd
import torch
import vec2text

TARGET_TEXT_FILE  = "data/sentences_target_text_db.parquet"
TARGET_INDEX_FILE = "data/sentences_target_gtr_vector_db.index"
GTR_MODEL_NAME    = "sentence-transformers/gtr-t5-base"
NUM_STEPS         = 20
PREFER_GPU        = True


def normalize_text(text: str) -> str:
    return " ".join(str(text).split())


def reconstruct_embedding(row_id: int, device: str) -> torch.Tensor:
    return torch.tensor(target_fi.reconstruct(int(row_id)), dtype=torch.float32).unsqueeze(0).to(device)

**Load data and models.** Load the GTR-encoded target embeddings and the pretrained vec2text corrector. `load_gtr_encoder` is provided in `gtr_runtime.py`.

In [ ]:
target_df = pd.read_parquet(TARGET_TEXT_FILE).sort_values("id").reset_index(drop=True)
target_fi = faiss.read_index(TARGET_INDEX_FILE)

assert len(target_df) == target_fi.ntotal, "Parquet and FAISS are out of sync — re-run 04_1."

print(f"{len(target_df)} sentences | FAISS dim={target_fi.d}")

gtr_encoder, gtr_device = load_gtr_encoder(GTR_MODEL_NAME, prefer_gpu=PREFER_GPU)
corrector = vec2text.load_pretrained_corrector("gtr-base")
corrector_device = next(corrector.model.parameters()).device
print(f"GTR on {gtr_device} | corrector on {corrector_device}")

**Attack.** Invert each target embedding. For each result, re-encode the reconstructed sentence and measure cosine similarity against the original — this is your fidelity metric.

In [ ]:
results = []

for row in tqdm(target_df.itertuples(index=False), total=len(target_df)):
    target_emb = reconstruct_embedding(row.id, str(corrector_device))
    t0 = perf_counter()
    reconstructed = ...
    elapsed = perf_counter() - t0
    cosine = ...
    results.append({
        "target_id":    row.target_id,
        "ground_truth": row.text,
        "reconstructed": reconstructed,
        "cosine":       cosine,
        "exact_match":  normalize_text(row.text) == normalize_text(reconstructed),
    })
    print(f"[{len(results):>2}/{len(target_df)}] {row.target_id:<12} cosine={cosine:.3f} | {normalize_text(reconstructed)!r}")

**Results.** Summarize attack performance: mean cosine similarity and exact-match rate. How many sentences were fully recovered?

In [ ]:
cosines = [r["cosine"] for r in results]
exact_matches = sum(r["exact_match"] for r in results)
print(f"Mean cosine : {sum(cosines)/len(cosines):.3f} | Exact matches : {exact_matches}/{len(results)}\n")

for r in results:
    print("------")
    print(r["target_id"])
    print(f"original: {r['ground_truth']}")
    print(f"reconstructed: {normalize_text(r['reconstructed'])}")
    print(f"cosine: {r['cosine']:.3f}")
print("------")

**Exploration.** Repeat the inversion on a single target using 1, 5, 10, and 20 refinement steps. Where does quality plateau?

In [ ]:
focus_row = target_df.iloc[0]
focus_emb = reconstruct_embedding(focus_row["id"], str(corrector_device))
print(f"Target : {focus_row['text']!r}\n")

for steps in [1, 5, 10, 20]:
    ...